# Project: Sales Lead Triage Agent
**Brief from Paras:**
Pulls new leads from Gmail, enriches via web research, scores them against an ICP in Notion,
books a discovery call on Calendar for hot leads or drops them in Slack for the SDR.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class SalesLeadTriageState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    lead_email: str
    lead_company: str
    enrichment_data: Optional[dict]
    icp_score: Optional[int]
    tier: Optional[str]
    calendar_link: Optional[str]


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# enricher - Tavily web research
enricher_tools = toolset.get_tools(actions=[Action.TAVILY_TAVILY_SEARCH, Action.TAVILY_TAVILY_EXTRACT])
enricher_agent = create_react_agent(llm, enricher_tools, prompt='''Research the lead's company. Return company size, industry, recent news.''')
def enricher_node(state):
    result = enricher_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='enricher')]}


# scorer - Notion ICP scorer
scorer_tools = toolset.get_tools(actions=[Action.NOTION_QUERY_DATABASE])
scorer_agent = create_react_agent(llm, scorer_tools, prompt='''Look up our ICP rules in Notion and score the lead 0-100. Set tier hot|warm|cold.''')
def scorer_node(state):
    result = scorer_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='scorer')]}


# booker - Calendar booker
booker_tools = toolset.get_tools(actions=[Action.GOOGLECALENDAR_CREATE_EVENT])
booker_agent = create_react_agent(llm, booker_tools, prompt='''If tier is hot, book a 30-min discovery call. Return calendar link.''')
def booker_node(state):
    result = booker_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='booker')]}


# notifier - Slack notifier
notifier_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
notifier_agent = create_react_agent(llm, notifier_tools, prompt='''Post lead summary to #sales-leads channel.''')
def notifier_node(state):
    result = notifier_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='notifier')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if state.get('enrichment_data') is None: return {'next_worker': 'enricher'}
    if state.get('icp_score') is None: return {'next_worker': 'scorer'}
    if state.get('tier') == 'hot' and state.get('calendar_link') is None: return {'next_worker': 'booker'}
    if 'notified' not in (state['messages'][-1].content.lower() if state['messages'] else ''):
        return {'next_worker': 'notifier'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'scorer', 'booker', 'notifier', 'enricher'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("enricher", enricher_node)
g.add_node("scorer", scorer_node)
g.add_node("booker", booker_node)
g.add_node("notifier", notifier_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "enricher": "enricher",
    "scorer": "scorer",
    "booker": "booker",
    "notifier": "notifier",
    "__end__": END,
})
g.add_edge("enricher", "supervisor")
g.add_edge("scorer", "supervisor")
g.add_edge("booker", "supervisor")
g.add_edge("notifier", "supervisor")

conn = sqlite3.connect("01_sales_lead_triage.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '01_sales_lead_triage-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'lead_email': 'vp.eng@acme.io', 'lead_company': 'Acme Corp',
        'messages': [HumanMessage(content='New lead: vp.eng@acme.io from Acme Corp')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
